# Cycle-space certificates: reproducibility notebook

This notebook is the numerical companion to **When Gigawatts of Computational Load Disappear: Cycle-Space Certificates for Grid Synchronization and Transient Stability**.

The model uses MATPOWER case39/case118 topology and solved voltage magnitudes, reduced to a lossless fixed-voltage network. The transient experiment is an **instantaneously balanced injection step**

$$p_{\rm eff}=p+\Delta P(e_\delta-\gamma),\qquad \mathbf 1^\top\gamma=1,$$

not a raw load rejection followed by a governor/droop transient. Fresh outputs are written to `results/generated/`; checked-in manuscript values live separately in `results/reference/`.


In [ ]:
from pathlib import Path
import os, sys, json, numpy as np
import matplotlib.pyplot as plt

# Make the notebook work whether Jupyter was started in the repository root
# or inside notebooks/.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from grids import load_case39, load_case118
from syncnet import *
from transient import *
from exp1_theory import validation, ring_counts, static_case
from exp2_transient import main as run_transient
from figures import make_figures
from paper_results import STATIC, CONCENTRATIONS, TRANSIENT_MW

Path('results/generated').mkdir(parents=True, exist_ok=True)
Path('figures/generated').mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=7, suppress=True)
print('repository root:', ROOT)


## 1. Load the two MATPOWER reductions

On first use the loader downloads the standard MATPOWER 8.1 `case39.m` and `case118.m` files into `data/`. For an offline run, place those two files there manually.

Parallel branches are merged, so the reductions have 46 unique edges for case39 and 179 for case118.

In [ ]:
case39 = load_case39()
case118 = load_case118()
for name, case in [('case39', case39), ('case118', case118)]:
    G=case['G']
    print(name, 'n=',G.number_of_nodes(), 'L=',G.number_of_edges(),
          'c=',G.number_of_edges()-G.number_of_nodes()+1,
          'sum(p)=',case['p'].sum())


## 2. Numerical checks of the cycle-space convex program

We check finite-difference derivatives, reconstruct node angles from the dual solution, verify tree exactness implicitly through the static formulas, and reproduce the closed-form winding count on rings:
\[
N_{\rm winding}(n)=2\left\lfloor\frac{n-1}{4}\right\rfloor+1.
\]

In [ ]:
val = validation()
val


In [ ]:
ring_n, ring_count, ring_pred = ring_counts()
print('all ring counts correct:', np.all(ring_count == ring_pred))
plt.figure(figsize=(6.2,3.2))
plt.plot(ring_n, ring_count, 'o', label='cycle-space programs')
plt.plot(ring_n, ring_pred, '-', label=r'$2\lfloor(n-1)/4\rfloor+1$')
plt.xlabel('ring size n'); plt.ylabel('strict-cohesion winding states')
plt.legend(frameon=False); plt.tight_layout(); plt.show()


## 3. Static loading thresholds

For a scaled injection vector `alpha * p`, the DCB statistic scales linearly. The exact zero-winding strict-cohesion limit is obtained from the cycle-space convex program. A separate warm-started primal continuation is followed beyond the cohesive boundary as long as Newton converges to a locally stable state (`lambda_2 > 0`). The latter is only a **numerical branch limit**, not a certified saddle-node.

In [ ]:
r39 = static_case(case39, 'case39')
r118 = static_case(case118, 'case118')
print(json.dumps(r118, indent=2))
print(json.dumps(r39, indent=2))

static_payload = {'case39': r39, 'case118': r118}
Path('results/generated/static_results.json').write_text(json.dumps(static_payload, indent=2))


In [ ]:
print('Paper values:')
print(json.dumps(STATIC, indent=2))
for case, r in [('case118',r118),('case39',r39)]:
    print(case, {q: r[q]-STATIC[case][q] for q in ['dcb','cohesive','continued']})


## 4. Case39 balanced-step transient experiment

At `alpha=4`, the most negative injection is MATPOWER bus 20. Removing that full load block corresponds to about 2724.5 MW after the lossless balancing projection. The fully concentrated simultaneous balancing action is placed at MATPOWER bus 38; concentration `0` spreads the action uniformly over all generator buses.

For each concentration we compute four thresholds:

1. energy-certified: `W0 < c*`;
2. finite-horizon RK4 cohesion (`T=20`, `dt=0.004`);
3. exact static strict-cohesion feasibility;
4. DCB static screen.

The threshold bisection is at approximately 1 MW resolution.

In [ ]:
# Expensive cell: all 92 signed cohesive boundary faces are solved repeatedly.
transient_payload = run_transient('results/generated', do_refinement_checks=False)


In [ ]:
rows_mw = np.asarray(transient_payload['rows_MW'])
print('columns: certified, simulated, static exact, static DCB [MW]')
print(rows_mw)
print('\npaper values:')
print(np.column_stack([TRANSIENT_MW[k] for k in ['certified','simulated','static_exact','static_dcb']]))


### Optional simulation refinement

The manuscript reports that halving the RK4 step to `dt=0.002` did not change the threshold at 1 MW resolution, and that extending the two extreme allocations to `T=30` also did not change the reported threshold. The next cell is optional because it adds extra simulation time.

In [ ]:
RUN_REFINEMENT = False
if RUN_REFINEMENT:
    _ = run_transient('results/generated_refined', do_refinement_checks=True)


## 5. Regenerate the compact paper figures

In [ ]:
make_figures('figures/generated',
             'results/generated/static_results.json',
             'results/generated/transient_results.json')
from IPython.display import Image, display
display(Image('figures/generated/static_thresholds.png'))
display(Image('figures/generated/transient_thresholds.png'))


## 6. Interpretation and numerical caveats

- The strict-cohesion limit is the supremum of an **open** cell; its last digits depend on numerical tolerance.
- The continued case118 branch extends beyond 90-degree line differences and is not a certified stability boundary.
- The critical-energy face decomposition is exact mathematically, but its face minima are floating-point numerical solves rather than interval-certified lower bounds.
- The transient model intentionally omits governor lags, voltage dynamics, protection, converter controls, and load dynamics.
- The finite-horizon simulation is a diagnostic; the theorem is the conservative energy certificate.